# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/youssef-mm/FlyRank-ML-Assignment/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Plain-Words Rule Description
**Selected Lane: Content Refresh & Prioritization (Lane 1)**
In this lane, the core objective is to decide which existing content pages should be refreshed to arrest performance decline and reclaim lost search visibility.

**The Rule in Plain Words:**
> "A page is prioritized for a refresh review if it represents substantial search visibility (`impressions_90d >= 500`) and has aged without an editorial update (`days_since_last_update >= 90`), indicating decaying content relevance on high-exposure queries."

### Two Signal Audits (Pre-Rule Verification)
Before building the rule, we audit two candidate signals from observable data:

1. **Signal 1 (Flag-Linked: Staleness behind refresh flags):**
   - *Hypothesis:* Content staleness (`days_since_last_update` or `freshness_tier`) is positively associated with traffic decline (`is_declining_label`).
   - *Audit Finding:* Across active content tiers, pages unrefreshed for 3–6 months (`91-180`) show a **61.1%** decline rate compared to **51.1%** for recently refreshed pages (`0-30`), an increase of +10.0 percentage points. However, extreme staleness (`181+`) shows a drop in decline rate to **47.1%** because those 174 rows represent dead, near-zero traffic pages (median impressions: 15.5) that have already flatlined.
   - *Verdict:* **MIXED** (unconditioned) / **CONFIRMED** (with a volume floor). This clearly-explained nuance saves our rule: staleness is a valid decay signal only when paired with a minimum visibility floor.

2. **Signal 2 (Flag-Linked: CTR-vs-Position behind CTR-fix logic):**
   - *Hypothesis:* Pages ranking on Page 1 (`position_tier == 'page_1'`) with below-average CTR (<0.10%) experience higher decline rates, signaling user intent mismatch or snippet decay.
   - *Audit Finding:* On Page 1 with volume floor (`impressions_90d >= 100`), pages with critical low CTR (<0.10%) decline at a rate of **74.1%**, while pages with healthy CTR (>=0.50%) decline at **49.3%** (a 24.8 percentage point gap).
   - *Verdict:* **CONFIRMED**. A severe CTR deficit on Page 1 strongly correlates with active decline.

### The Rule's Reason Codes and Action Labels
- **Reason Codes:**
  - `stale_visible_decay_risk`: Assigned to pages satisfying both the staleness threshold (`days_since_last_update >= 90`) and visibility threshold (`impressions_90d >= 500`).
  - `stable_or_fresh`: Assigned to pages that are recently updated or lack sufficient visibility to justify editorial intervention.
- **Action Labels:**
  - `refresh_content`: Actionable recommendation to schedule page for editorial rewrite, keyword re-optimization, and timestamp refresh.
  - `monitor`: Passive observation; no immediate content intervention needed.

In [1]:
import os
import numpy as np
import pandas as pd

# Load dataset (local path with Colab raw URL fallback)
data_path = "data/raw/content_refresh_anonymized.csv"
if not os.path.exists(data_path):
    data_path = "../../data/raw/content_refresh_anonymized.csv"
if not os.path.exists(data_path):
    data_path = "https://raw.githubusercontent.com/youssef-mm/FlyRank-ML-Assignment/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(data_path)
print(f"Loaded dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")

# Ground truth outcome (for signal testing and precision evaluation only - NOT a feature)
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)
base_rate = df["is_declining_label"].mean()
print(f"Overall dataset base rate (is_declining_label): {base_rate:.3%}\n")

# --- Signal Test 1: Staleness (freshness_tier) behind refresh flags ---
print("=" * 70)
print("SIGNAL 1 TEST: Staleness (freshness_tier) behind refresh flags")
print("=" * 70)
s1 = df.groupby("freshness_tier").agg(
    n=("is_declining_label", "count"),
    declining_count=("is_declining_label", "sum"),
    declining_rate=("is_declining_label", "mean"),
    median_impressions=("impressions_90d", "median")
).reset_index()
s1["declining_rate_pct"] = (s1["declining_rate"] * 100).round(1)
print(s1[["freshness_tier", "n", "declining_count", "declining_rate_pct", "median_impressions"]].to_string(index=False))

# Volume-floor check on Staleness (impressions_90d >= 100)
s1_floor = df[df["impressions_90d"] >= 100].groupby("freshness_tier").agg(
    n=("is_declining_label", "count"),
    declining_rate=("is_declining_label", "mean"),
    median_impressions=("impressions_90d", "median")
).reset_index()
s1_floor["declining_rate_pct"] = (s1_floor["declining_rate"] * 100).round(1)
print("\n[Floor Check] With impressions_90d >= 100:")
print(s1_floor[["freshness_tier", "n", "declining_rate_pct", "median_impressions"]].to_string(index=False))
print("\nSignal 1 Verdict: MIXED")
print("Explanation: Decline rate increases from 51.1% (0-30d) to 61.1% (91-180d), but drops at 181+ (47.1%)")
print("because unconditioned 181+ rows are dormant zombie pages (median 15.5 impressions) that already bottomed out.")

# --- Signal Test 2: CTR-vs-Position on Page 1 (CTR-fix logic) ---
print("\n" + "=" * 70)
print("SIGNAL 2 TEST: CTR-vs-Position on Page 1 (CTR-fix logic)")
print("=" * 70)
p1 = df[(df["position_tier"] == "page_1") & (df["impressions_90d"] >= 100)].copy()
p1["ctr_band"] = pd.cut(
    p1["ctr"],
    bins=[-0.001, 0.10, 0.25, 0.50, 100.0],
    labels=["<0.10% (Critical Low)", "0.10%-0.25% (Low)", "0.25%-0.50% (Average)", ">=0.50% (Healthy)"]
)
s2 = p1.groupby("ctr_band", observed=False).agg(
    n=("is_declining_label", "count"),
    declining_count=("is_declining_label", "sum"),
    declining_rate=("is_declining_label", "mean"),
    median_ctr=("ctr", "median"),
    median_impressions=("impressions_90d", "median")
).reset_index()
s2["declining_rate_pct"] = (s2["declining_rate"] * 100).round(1)
print(s2[["ctr_band", "n", "declining_count", "declining_rate_pct", "median_ctr", "median_impressions"]].to_string(index=False))
print("\nSignal 2 Verdict: CONFIRMED")
print("Explanation: Pages on Page 1 with critical low CTR (<0.10%) decline at 74.1% vs 49.3% for healthy CTR (>=0.50%).")


Loaded dataset: 30,000 rows x 44 columns
Overall dataset base rate (is_declining_label): 54.207%

SIGNAL 1 TEST: Staleness (freshness_tier) behind refresh flags
freshness_tier     n  declining_count  declining_rate_pct  median_impressions
          0-30 20480            10473                51.1               470.0
          181+   174               82                47.1                15.5
         31-90   175              103                58.9               510.0
        91-180  9171             5604                61.1              1692.0

[Floor Check] With impressions_90d >= 100:
freshness_tier     n  declining_rate_pct  median_impressions
          0-30 13735                58.3              1450.0
          181+    35                74.3               429.0
         31-90   152                59.2               688.0
        91-180  8084                62.2              2286.0

Signal 1 Verdict: MIXED
Explanation: Decline rate increases from 51.1% (0-30d) to 61.1% (91-180d), 

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

### Rule Encoding & Queue Design
Following the live session methodology and the `building-baselines` skill:
1. **Readable Score Formula:**
   $$\text{stale} = (\text{days\_since\_last\_update} \ge 90)$$
   $$\text{visible} = (\text{impressions\_90d} \ge 500)$$
   $$\text{baseline\_refresh\_score} = \text{stale} \times \text{visible} \times \text{impressions\_90d}$$

2. **Reason Code & Action:**
   - Pages meeting the criteria receive reason code `stale_visible_decay_risk` and suggested action `refresh_content`.
   - All other pages receive `stable_or_fresh` and action `monitor`.

3. **No Leakage Guarantee:**
   - The score utilizes strictly observable pre-decision inputs (`days_since_last_update`, `impressions_90d`).
   - Neither `trend_pct`, `trend_direction`, nor `is_declining_label` are touched during feature creation or scoring.

4. **Honest Baseline Evaluation:**
   - Evaluated using **Precision@K** ($K \in \{10, 20, 50, 100\}$) compared against the overall dataset base rate (54.21%).
   - The full ranked queue is exported to `work/outputs/baseline_action_score.csv`.
   - Run receipts are saved to `work/outputs/baseline_metrics.json`.

In [2]:
import json
from pathlib import Path

# 1. Encode the transparent baseline rule
stale = (df["days_since_last_update"] >= 90).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)

df["baseline_refresh_score"] = stale * visible * df["impressions_90d"]
df["baseline_rank"] = df["baseline_refresh_score"].rank(method="first", ascending=False).astype(int)
df["reason_code"] = np.where(df["baseline_refresh_score"] > 0, "stale_visible_decay_risk", "stable_or_fresh")
df["action_label"] = np.where(df["baseline_refresh_score"] > 0, "refresh_content", "monitor")

# 2. Precision@K evaluation function
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

p10 = precision_at_k(df["baseline_refresh_score"], df["is_declining_label"], 10)
p20 = precision_at_k(df["baseline_refresh_score"], df["is_declining_label"], 20)
p50 = precision_at_k(df["baseline_refresh_score"], df["is_declining_label"], 50)
p100 = precision_at_k(df["baseline_refresh_score"], df["is_declining_label"], 100)

print("=" * 60)
print("BASELINE EVALUATION METRICS (Precision@K)")
print("=" * 60)
print(f"Dataset Base Rate       : {base_rate:.3%}")
print(f"Precision@10            : {p10:.3%} (lift: {p10/base_rate:.2f}x)")
print(f"Precision@20            : {p20:.3%} (lift: {p20/base_rate:.2f}x)")
print(f"Precision@50            : {p50:.3%} (lift: {p50/base_rate:.2f}x)")
print(f"Precision@100           : {p100:.3%} (lift: {p100/base_rate:.2f}x)")
print(f"Flagged for Refresh     : {(df['baseline_refresh_score'] > 0).sum():,} pages")

# 3. Write ranked queue and metrics
out_dir = Path("work/outputs")
if not out_dir.exists() and Path("../outputs").exists():
    out_dir = Path("../outputs")
out_dir.mkdir(parents=True, exist_ok=True)

csv_path = out_dir / "baseline_action_score.csv"
queue_cols = [
    "baseline_rank", "content_id", "client_id", "baseline_refresh_score",
    "reason_code", "action_label", "impressions_90d", "days_since_last_update",
    "avg_position", "ctr", "is_declining_label"
]
queue_df = df.sort_values("baseline_rank")[queue_cols]
queue_df.to_csv(csv_path, index=False)
print(f"\n[OK] Ranked queue written to: {csv_path} ({len(queue_df):,} rows)")

metrics = {
    "base_rate": float(base_rate),
    "precision_at_10": float(p10),
    "precision_at_20": float(p20),
    "precision_at_50": float(p50),
    "precision_at_100": float(p100),
    "total_queue_rows": len(queue_df),
    "flagged_refresh_count": int((df["baseline_refresh_score"] > 0).sum())
}
metrics_path = out_dir / "baseline_metrics.json"
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"[OK] Metrics JSON written to: {metrics_path}")


BASELINE EVALUATION METRICS (Precision@K)
Dataset Base Rate       : 54.207%
Precision@10            : 60.000% (lift: 1.11x)
Precision@20            : 45.000% (lift: 0.83x)
Precision@50            : 44.000% (lift: 0.81x)
Precision@100           : 38.000% (lift: 0.70x)
Flagged for Refresh     : 6,575 pages

[OK] Ranked queue written to: work\outputs\baseline_action_score.csv (30,000 rows)
[OK] Metrics JSON written to: work\outputs\baseline_metrics.json


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Below is the line-by-line review of the top ten recommendations from our baseline queue, analyzing the action, why it scored, and what failure mode or confounding factor would make the recommendation wrong:

1. **Rank 1 (`content_5fe46e04994d` | `client_4e07408562`)**:
   - **Action:** `refresh_content`
   - **Why it's there:** Highest volume page in the stale cohort (517,715 impressions), 104 days since update, Page 1 position (4.2), low CTR (0.14%); actively declining (`is_declining_label = 1`).
   - **What would make it wrong:** If the low CTR is caused by dominant Google SERP features (Featured Snippets, Knowledge Panels) rather than decaying title/copy relevance.

2. **Rank 2 (`content_2dba2b1f9536` | `client_6208ef0f77`)**:
   - **Action:** `refresh_content`
   - **Why it's there:** Massive volume (443,434 impressions) and 104 days stale, triggering the exposure multiplier.
   - **What would make it wrong:** The page ranks deep on Page 3 (avg position 27.9) and is currently NOT declining (`is_declining_label = 0`). Editorial copy refreshes cannot solve fundamental domain authority or backlink deficits for Page 3 queries.

3. **Rank 3 (`content_2c2606c5d176` | `client_19581e27de`)**:
   - **Action:** `refresh_content`
   - **Why it's there:** Top Page 1 position (4.2) with 347,399 impressions, moderate CTR (0.53%), 104 days stale; actively declining (`is_declining_label = 1`).
   - **What would make it wrong:** If the decline is seasonal search query contraction across the industry rather than content fatigue.

4. **Rank 4 (`content_cb112fce36be` | `client_19581e27de`)**:
   - **Action:** `refresh_content`
   - **Why it's there:** Strong Page 1 position (5.6), 309,910 impressions, weak CTR (0.16%), 104 days stale; actively declining (`is_declining_label = 1`).
   - **What would make it wrong:** If the page targets broad navigational queries where search intent is fixed and editorial rewrites risk dropping rank out of Page 1.

5. **Rank 5 (`content_9532f197bbc8` | `client_4e07408562`)**:
   - **Action:** `refresh_content`
   - **Why it's there:** Prime position 2.0 with high CTR (0.87%), 309,192 impressions, 104 days stale; actively declining (`is_declining_label = 1`).
   - **What would make it wrong:** Rewriting a position #2 page carries substantial catastrophic disruption risk; aggressive changes may cause Google to re-evaluate the URL and lose prime placement.

6. **Rank 6 (`content_36ff89c8214e` | `client_19581e27de`)**:
   - **Action:** `refresh_content`
   - **Why it's there:** High exposure (295,097 impressions), 104 days stale, position 7.3 with very low CTR (0.05%).
   - **What would make it wrong:** The page is NOT declining (`is_declining_label = 0`). Modifying stable copy could destabilize a page that is successfully holding ranking.

7. **Rank 7 (`content_b28d1efd668f` | `client_6208ef0f77`)**:
   - **Action:** `refresh_content`
   - **Why it's there:** High impression volume (286,608 impressions), 104 days stale.
   - **What would make it wrong:** Average position is 26.2 (Page 3) and performance is stable (`is_declining_label = 0`). Editorial resources spent on Page 3 content have low conversion leverage.

8. **Rank 8 (`content_813e88069237` | `client_6208ef0f77`)**:
   - **Action:** `refresh_content`
   - **Why it's there:** High volume (233,561 impressions), 104 days stale, actively declining (`is_declining_label = 1`).
   - **What would make it wrong:** Average position is deep (26.2). Without internal links or site-wide authority gains, an on-page copy refresh is unlikely to recover lost visibility.

9. **Rank 9 (`content_c21024970297` | `client_19581e27de`)**:
   - **Action:** `refresh_content`
   - **Why it's there:** Strong Page 1 position (5.1), 211,366 impressions, 104 days stale.
   - **What would make it wrong:** The page is NOT declining (`is_declining_label = 0`). Flagging healthy, stable pages frustrates content teams and wastes editorial budgets.

10. **Rank 10 (`content_c8e9d6ab9013` | `client_19581e27de`)**:
    - **Action:** `refresh_content`
    - **Why it's there:** Page 1 position (9.7), 208,678 impressions, 0.00% measured CTR, 104 days stale; actively declining (`is_declining_label = 1`).
    - **What would make it wrong:** The 0.00% CTR indicates an informational query completely answered in SERP (zero-click query) where clicks cannot be recovered by updating the article.

In [3]:
# Display top 10 ranked queue items with diagnostic fields
top10_df = queue_df.head(10)[[
    "baseline_rank", "content_id", "client_id", "baseline_refresh_score",
    "action_label", "reason_code", "impressions_90d", "days_since_last_update",
    "avg_position", "ctr", "is_declining_label"
]]
print("TOP 10 BASELINE RECOMMENDATIONS:")
print(top10_df.to_string(index=False))

print("\n" + "-" * 70)
print("Top 10 Summary:")
print(f"Actually Declining (True Positives): {top10_df['is_declining_label'].sum()} / 10 ({top10_df['is_declining_label'].mean():.1%})")
print(f"Stable / Non-Declining (False Positives): {(top10_df['is_declining_label'] == 0).sum()} / 10")


TOP 10 BASELINE RECOMMENDATIONS:
 baseline_rank           content_id         client_id  baseline_refresh_score    action_label              reason_code  impressions_90d  days_since_last_update  avg_position  ctr  is_declining_label
             1 content_5fe46e04994d client_4e07408562                  517715 refresh_content stale_visible_decay_risk           517715                     104           4.2 0.14                   1
             2 content_2dba2b1f9536 client_6208ef0f77                  443434 refresh_content stale_visible_decay_risk           443434                     104          27.9 0.21                   0
             3 content_2c2606c5d176 client_19581e27de                  347399 refresh_content stale_visible_decay_risk           347399                     104           4.2 0.53                   1
             4 content_cb112fce36be client_19581e27de                  309910 refresh_content stale_visible_decay_risk           309910                     104           5

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak Picks Analysis (Where the Baseline Fails)
Hand-reviewing the top of the ranked queue reveals the structural shortcomings of a simple rule-based score:
1. **The Raw Volume Trap (False Positives on Deep Positions):**
   - **Rank 2 (`content_2dba2b1f9536`)** and **Rank 7 (`content_b28d1efd668f`)** scored high purely because of heavy raw impression volume (443k and 286k impressions). However, both pages sit on Page 3 (average position 27.9 and 26.2) and are **not declining** (`is_declining_label = 0`). The rule treats all impressions equally, failing to penalize deep positions where clicks are practically zero regardless of content age.
2. **The Healthy Page Trap (Intervention Risk):**
   - **Rank 6 (`content_36ff89c8214e`)** and **Rank 9 (`content_c21024970297`)** rank strongly on Page 1 (positions 7.3 and 5.1) and are completely stable (`is_declining_label = 0`). Editing high-ranking stable pages risks disrupting existing keyword equity and causing actual declines.
3. **What This Means for the Week-5 Model:**
   - The machine learning model in Week 5 must learn multi-variable non-linear interactions (e.g., conditioning on position opportunity, engagement rate, and CTR deficit relative to position) rather than relying on linear volume multipliers.

### Leakage Audit
We conducted a strict audit to ensure no data leakage was introduced:
- **Prohibited Target-Derived Features:** Neither `trend_direction`, `trend_pct`, nor `is_declining_label` were used as inputs to the score or ranking logic.
- **Product Flags:** No FlyRank product decisions (`health_score`, `priority_score`, `needs_ctr_fix`) were included.
- **Window Alignment:** The baseline utilizes only pre-decision trailing-90-day exposure metrics (`impressions_90d`) and metadata age (`days_since_last_update`).

In [4]:
# 1. Programmatic Leakage Verification
scoring_features = ["days_since_last_update", "impressions_90d"]
prohibited_substrings = ["trend_", "label", "is_declining", "health_score", "priority_score"]

for feat in scoring_features:
    for prohibited in prohibited_substrings:
        assert prohibited not in feat, f"LEAKAGE DETECTED: {feat} contains {prohibited}!"

print("[OK] Leakage check passed: No target-derived, product flag, or future-window features used in scoring.")

# 2. Output verification
assert csv_path.exists(), f"Missing output CSV: {csv_path}"
assert csv_path.stat().st_size > 0, "Output CSV is empty!"
assert metrics_path.exists(), f"Missing metrics JSON: {metrics_path}"

print(f"[OK] Output verification passed: {csv_path} ({csv_path.stat().st_size / 1024:.1f} KB) and {metrics_path} confirmed.")


[OK] Leakage check passed: No target-derived, product flag, or future-window features used in scoring.
[OK] Output verification passed: work\outputs\baseline_action_score.csv (2768.2 KB) and work\outputs\baseline_metrics.json confirmed.
